<a href="https://colab.research.google.com/github/Zeeba1705/atml_PA0/blob/main/Copy_of_task1_pa0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ATML PA0 - Task 1


Inner workings of Resnet-152

In [2]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

#delete after

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA available: True
Device: cuda


In [3]:
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import random
import time
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.models import ResNet152_Weights, resnet152

In [4]:
#so that there's consistency between runs

SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Preprocessing CIFAR-10: In order to match the input expected by Resnet-152


In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

100%|██████████| 170M/170M [35:48<00:00, 79.4kB/s]


In [7]:
train_size = 45000
val_size = 5000

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

**Loading Resnet-152:**

In [13]:
weights = ResNet152_Weights.DEFAULT
model = resnet152(weights=weights)

In [9]:
#final layer of Resnet-152
print(model.fc)

Linear(in_features=2048, out_features=1000, bias=True)


This is the final layer I need to replace, as currently the classifier produces scores for a 1000 classes (out_features=1000), whereas for CIFAR-10 we only need 10.

In [14]:
#freezing parameters first
for param in model.parameters():
    param.requires_grad = False

#now we replace the final layer, goign from 1000 to 10 as needed
model.fc = nn.Linear(2048, 10)
print(model.fc)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

#Optimizer: we r only updating the new classification layer
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

Linear(in_features=2048, out_features=10, bias=True)


In [11]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
  #just checking.. should get only fc.bias, fc.weight.

fc.weight
fc.bias


In [15]:
num_e= 3

for epoch in range(num_e):

    model.train()

    tr_loss= 0
    train_correct= 0
    train_total= 0

    for images, labels in train_loader:

        images= images.to(device)
        labels= labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss= criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        tr_loss+= loss.item() * images.size(0)

        predicted= outputs.argmax(dim=1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    tr_loss= tr_loss / train_total
    train_accuracy= train_correct / train_total

    model.eval()

    val_loss= 0
    val_correct= 0
    val_total= 0

    with torch.no_grad():

        for images, labels in val_loader:

            images= images.to(device)
            labels= labels.to(device)

            outputs= model(images)

            loss= criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predicted= outputs.argmax(dim=1)

            val_total+= labels.size(0)
            val_correct+= (predicted == labels).sum().item()

    val_loss= val_loss / val_total
    val_accuracy= val_correct / val_total


    print(
        f"Epoch {epoch + 1}/{num_e} | "
        f"Train Loss: {tr_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch 1/3 | Train Loss: 0.7207 | Train Accuracy: 0.7800 | Val Loss: 0.5228 | Val Accuracy: 0.8242
Epoch 2/3 | Train Loss: 0.5000 | Train Accuracy: 0.8340 | Val Loss: 0.4734 | Val Accuracy: 0.8410
Epoch 3/3 | Train Loss: 0.4497 | Train Accuracy: 0.8492 | Val Loss: 0.4549 | Val Accuracy: 0.8438
